# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

# Step 1: Install required packages

In [1]:
%pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken

   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   ---------------------------------------- 549.1/549.1 kB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 11.1 MB/s  0:00:00
   ---------------------------------------- 0.0/918.7 kB ? eta -:--:--
   ---------------------------------------- 918.7/918.7 kB 10.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 9.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------  2.4/2.4 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 10.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 10.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] T

# Step 2: Import dependencies

In [22]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from dotenv import load_dotenv
#update

# Step 3: Set Azure OpenAI credentials



In [23]:
# Load environment variables from .env file
load_dotenv()

True

# Step 4: Load and chunk your custom FAQ document


In [7]:
# Load the FAQ document and split it into chunks for embedding

loader = TextLoader("faq.txt")  # Ensure this file exists
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)
print(f"Number of document chunks: {len(docs)}")

Number of document chunks: 4


# Step 5: Create vectorstore using Azure embeddings


In [12]:
# Embed document chunks and store them in a FAISS vector index

embeddings = OpenAIEmbeddings(
   
    api_version="2023-05-15",
    deployment="text-embedding-ada-002",
    api_key=os.environ["OPENAI_API_KEY"]
)

vectorstore = FAISS.from_documents(docs, embeddings)

# Step 6: Initialize the Azure OpenAI LLM

In [24]:
# Initialize the GPT-4o model from Azure with temperature 0 for deterministic output

llm = ChatOpenAI(    
    model="gpt-5-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)
#update

# Step 7: Create the RAG chain

In [25]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Step 8: Ask a question

In [26]:
# Send a question to the RAG chain and store the result

query = "What is our return policy?"
result = qa_chain.invoke({"query": query})

# Step 9: Print results

In [27]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

Answer: Our return policy: you can return unopened items within 30 days. Products must be returned in their original packaging and include proof of purchase.

--- Sources ---

Source 1:
Q: What is your return policy?
A: We offer a 30-day return policy for all unopened items. Products must be returned in original packaging with proof of purchase.

Q: Do you offer international shipping?
A: Yes, we ship internationally to over 50 countries. Delivery times and shipping costs vary by destination.

Q: How can I contact customer support?
A: You can reach us via email at support@example.com or call our helpline at +1-800-123-4567. Our support hours are 9AM to 6PM, Monday to Friday.

Source 2:
Q: What payment methods are accepted?
A: We accept Visa, MasterCard, American Express, PayPal, and UPI payments.

Q: How can I cancel my order?
A: Orders can be cancelled within 2 hours of placement. Log in to your account, go to "My Orders", and click on "Cancel Order".

Q: Is there a warranty on produc